# Thin MCP Client with [Docker MCP Toolkit](https://docs.docker.com/ai/mcp-catalog-and-toolkit/toolkit/)

Anton Antonov   
May 2026

---

## Introduction

This notebook has a complete usage example of a _thin_ Model Context Protocol (MCP) client of the Raku package ["MCP::Client"](https://raku.land/zef:antononcube/MCP::Client), [AAp1].

The MCP server is run in Docker -- see ["Docker MCP Toolkit"](https://docs.docker.com/ai/mcp-catalog-and-toolkit/toolkit/).

"MCP::Client" provides the class `MCP::Client` which creates from MCP server's methods `LLM::Tool` objects that can be used with Raku's Large Language Model (LLM) framework implemented with ["LLM::Functions"](https://raku.land/zef:antononcube/LLM::Functions), ["LLM::Prompts"](https://raku.land/zef:antononcube/LLM::Prompts), ["Text::SubParsers"](https://raku.land/zef:antononcube/Text::SubParsers); see [AA3÷6, AAp1÷3]. 

*Because the client is thin and its implementation concise, it is designed to allow both (i) quick, "on-the-spot" MCP utilization and (ii) easy understanding of MCP principles.*

**Remark:** Similar workflow based on a "simple" Python MCP server is given in the file ["MCP-client-demo.raku"](https://github.com/antononcube/Raku-MCP-Client/blob/main/examples/MCP-client-demo.raku) and corresponding notebook ["Thin-MCP-client-demo.ipynb"](https://github.com/antononcube/Raku-MCP-Client/blob/main/docs/Thin-MCP-client-demo.ipynb).

**Remark:** The Wolfram Language (WL) paclet ["MCPClient"](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/MCPClient/), [AAp2], has the same mission of the (Raku) package "MCP::Client", but WL's "MCPClient" uses a Functional Programming implementation instead of an Object-Oriented Programming one (as "MCP::Client" does.)

---

## Setup

Load the packages used below.

In [12]:
use LLM::Functions;
use LLM::Tooling;
use LLM::Prompts;
use Text::SubParsers;

use Data::Translators;
use Data::Importers;
use JSON::Fast;

use MCP::Client;

---

## MCP Setup and initialization

Get a Docker profile file and show profile's name and identifier:

In [11]:
my $profile = data-import("https://raw.githubusercontent.com/antononcube/WL-MCPClient-paclet/refs/heads/main/Resources/default_docker_profile.json");
$profile<name id>.raku

("Default Docker Profile", "default-docker-profile")

Import the profile (to Docker):

In [ ]:
my $profileFile = $*TMPDIR ~ '/dockerProfile.json';
spurt($profileFile, to-json($profile));

my @cmd = "/usr/local/bin/docker", "mcp", "profile", "import", $profileFile;

my $proc = run @cmd, :out, :err;
$proc.out.slurp(:close).say;
$proc.err.slurp(:close).say;

Imported profile default-docker-profile




Set the PATH variable (MacOSX in this example):

In [14]:
for </usr/local/bin /usr/local/share/bin /usr/local/sbin> {
    %*ENV<PATH> = $_ ~ ':' ~ %*ENV<PATH> unless %*ENV<PATH>.contains($_)
}

()

Setup MCP server process creation command elements:

In [15]:
sink my @cmd = 'docker', 'mcp', 'gateway', 'run', '--profile', $profile<id>

Create the MCP client object:

In [16]:
my Bool:D $echo = False;
my Numeric:D $sleep = 5;
my $client = MCP::Client.new(:$echo, :$sleep);

sink $client.start(@cmd);

Initialize the client:

In [17]:
$client.initialize;

True

Instead of using the Docker profile ingestion and loading, using Docker's dashboard a default MCP Toolkit profile can be made and the just used the command `docker mcp gateway run`. Here is how such default profile looks like:

![](./img/Docker-dashboard-Default-MCP-profile.png)

---

## Tools discovery and creation

Get the MCP server tools list:

In [25]:
my @mcp-tools = |$client.list-tools();
@mcp-tools.elems

75

Randomly pick some tools and make a table of their tool records using only names and descriptions:

In [26]:
#% html
@mcp-tools.pick(5).sort(*<name>)
==> to-html(field-names => <name description>, align => 'left')

name,description
browser_fill_form,Fill multiple form fields
browser_select_option,Select an option in a dropdown
create_repository,Create a new GitHub repository in your account or specified organization
get_commit,Get details for a commit from a GitHub repository
get_latest_release,Get the latest release in a GitHub repository


Show tools of the official GitHub MCP server:

In [27]:
#% html
@mcp-tools.grep(*<description>.contains('GitHub',:i)) 
==> to-html(field-names => <name description>, align => 'left')

name,description
add_issue_comment,"Add a comment to a specific issue in a GitHub repository. Use this tool to add comments to pull requests as well (in this case pass pull request number as issue_number), but only if user is not asking specifically to add review comments."
assign_copilot_to_issue,Assign Copilot to a specific issue in a GitHub repository. This tool can help with the following outcomes: - a Pull Request created with source code changes to resolve the issue More information can be found at: - https://docs.github.com/en/copilot/using-github-copilot/using-copilot-coding-agent-to-work-on-tasks/about-assigning-tasks-to-copilot
create_branch,Create a new branch in a GitHub repository
create_or_update_file,"Create or update a single file in a GitHub repository. If updating, you should provide the SHA of the file you want to update. Use this tool to create or update a file in a GitHub repository remotely; do not use it for local file operations. In order to obtain the SHA of original file version before updating, use the following git command: git rev-parse <branch>:<path to file> SHA MUST be provided for existing file updates."
create_pull_request,Create a new pull request in a GitHub repository.
create_repository,Create a new GitHub repository in your account or specified organization
delete_file,Delete a file from a GitHub repository
fork_repository,Fork a GitHub repository to your account or specified organization
get_commit,Get details for a commit from a GitHub repository
get_file_contents,Get the contents of a file or directory from a GitHub repository


Make `LLM::Tool` objects:

In [28]:
my @tools = @mcp-tools.grep({ $_<description> ~~ /:i GitHub | YouTube/ }).map({ $client.to-llm-tool($_) });

deduce-type(@tools)

Vector((Any), 35)

Tools without properties:

In [29]:
.say for |@mcp-tools.grep({ !$_.<inputSchema><properties>  }).map(*<name description inputSchema>)

(browser_close Close the page {$schema => https://json-schema.org/draft/2020-12/schema, additionalProperties => False, properties => {}, type => object})
(browser_navigate_back Go back to the previous page in the history {$schema => https://json-schema.org/draft/2020-12/schema, additionalProperties => False, properties => {}, type => object})
(get_me Get details of the authenticated GitHub user. Use this when a request is about the user's own profile for GitHub. Or when information is missing to build other tool calls. {properties => {}, type => object})


Make a LLM access configuration with the tools:

In [30]:
my $conf = llm-configuration('ChatGPT', model => 'gpt-5.3-chat-latest', :@tools, :8192max-tokens);

LLM::Configuration(:name("chatgpt"), :model("gpt-5.3-chat-latest"), :module("WWW::OpenAI"), :max-tokens(8192))

--- 

## LLM invocations

Find the un-accepted GitHub Pull Requests (PRs) -- using the tool "list_pull_requests":

In [71]:
#% markdown
my $res = llm-synthesize(
    "Which of my -- antononcube -- GitHub pull requests are pending?", 
    e => $conf);

Right now you have **one open (pending) pull request**:

- Repo: ollama/ollama  
- PR #13778 — “Request to add community integration: Raku's "WWW::Ollama"”  
- Status: open  
- Created: 2026-01-19  
- Link: https://github.com/ollama/ollama/pull/13778  

If you expected more, it might be because:
- some were already merged or closed, or  
- they’re in repos/orgs with restricted visibility

Want me to dig up your recently merged or closed PRs too?

Create an LLM function that is assumed to invoke the GitHub MCP server tool "list_branches":

In [66]:
sink my &fGHB = -> $repo { 
    llm-synthesize([
            "Give the branches of the GitHub repo:",
            $repo, 
            'Separate the features from the bugfixes',
            llm-prompt('NothingElse')('JSON') 
        ],
        e => $conf,
        form => sub-parser('JSON'):drop
    )
}

**Remark:** Currently, `llm-function` of ["LLM::Functions"](https://raku.land/zef:antononcube/LLM::Functions/), [AAp3], does not support the use of LLM-tools, hence the "block-with-LLM-synthesis" definition.

Get branches data from a repository:

In [74]:
sink my $res = &fGHB('bduggan/raku-jupyter-kernel');

In [75]:
#% html
$res».subst(/ [ bugfix | feature ] '/' /) 
==> to-html(align=>'left')

bugfixes,trap_sigintwork-around-tty-issue
features,add_some_messagesadd-workflowsbetter_launcherdetermine_versionimplement_comm_info_requestmore_featuresraku-renamerename_executableupdate-readme


Using a tool of a different MCP server:

In [ ]:
# my $url = 'https://www.youtube.com/watch?v=-QtIVv-oz5Y';
# my $res = llm-synthesize(
#     "Get you get the transcript of this URL: $url",
#     e => $conf);

----

## Stopping the MCP server process

Kill the MCP client process:

In [ ]:
$client.kill;

----

## References

### Articles, blog posts

[AA1] Anton Antonov, ["LLM function calling workflows (Part 1, OpenAI)"](https://rakuforprediction.wordpress.com/2025/06/01/llm-function-calling-workflows-part-1-openai/), (2025), [RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA2] Anton Antonov, ["LLM function calling workflows (Part 2, Google's Gemini)"](https://rakuforprediction.wordpress.com/2025/06/01/llm-function-calling-workflows-part-2-google-gemini/), (2025), [RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA3] Anton Antonov, ["LLM function calling workflows (Part 3, Facilitation)"](https://rakuforprediction.wordpress.com/2025/06/08/llm-function-calling-workflows-part-3-facilitation/), (2025), [RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA4] Anton Antonov, ["LLM function calling workflows (Part 4, Universal specs)"](https://rakuforprediction.wordpress.com/2025/09/28/llm-function-calling-workflows-part-4-universal-specs/), (2025), [RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

### Packages

[AAp1] Anton Antonov, [MCP::Client, Raku package](https://github.com/antononcube/Raku-MCP-Client), (2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov, [MCPClient, Wolfram Language paclet](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/MCPClient/), (2026), [Wolfram Language Paclet Repository](https://resources.wolframcloud.com/PacletRepository).

[AAp3] Anton Antonov, [LLM::Functions, Raku package](https://github.com/antononcube/Raku-LLM-Functions), (2023-2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp4] Anton Antonov, [LLM::Prompts, Raku package](https://github.com/antononcube/Raku-LLM-Prompts), (2023-2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp5] Anton Antonov, [Text::SubParsers, Raku package](https://github.com/antononcube/Raku-Text-SubParsers), (2023), [GitHub/antononcube](https://github.com/antononcube).